# 古典模型：Colab GPU 試訓與評估

請選擇 GPU 並依序執行。現有六筆訓練資料僅供流程試驗，不能保證歷史限制或模型品質；下載底座需要網路、時間及足夠空間。


In [32]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "請先啟用 GPU 執行階段"
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

Thu Oct 29 12:05:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   68C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [33]:
!git clone --branch work --single-branch https://github.com/andrewlai1209-byte/BooleLOT.git /content/BooleLOT
%cd /content/BooleLOT
!git rev-parse HEAD

fatal: destination path '/content/BooleLOT' already exists and is not an empty directory.
/content/BooleLOT
eb971dc5536944456845807a76fcb6ae0ccbef18


## 安裝訓練依賴

依專案原有 Unsloth 範例安裝；版本或 GPU 相容性尚須實測。如安裝後要求重新啟動，請重啟並重新執行筆記本。

In [34]:
!pip install -q --upgrade uv
!uv pip install --system -q "torch>=2.8.0" "triton>=3.4.0" torchvision bitsandbytes "transformers==4.56.2" "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth" "git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels"
!uv pip install --system --no-deps "transformers==4.56.2" tokenizers "trl==0.22.2"

Using Python 3.13.16 environment at: /usr
Checked 3 packages in 104ms


In [35]:
!python -m training.prepare_corpus
!python -m training.train_classical --dry-run

{
  "catalogue_sha256": "34f331d4f04233987dea19c166d66cd9eb180bfb56ff9bb0784a4a684b847bf6",
  "train_count": 6,
  "eval_count": 2,
  "train_sources": [
    "15114-pdf.pdf",
    "pg5000-images-3.epub",
    "pg56386-images-3.epub",
    "pg70850-images-3.epub",
    "pg75391-images-3.epub",
    "tian_gong_kai_wu__0.zip"
  ],
  "eval_sources": [
    "pg68907-images-3.epub",
    "pg77167-images-3.epub"
  ],
  "note": "少量人工札記僅供流程試驗，不能代表完整古籍或足以驗證程式能力。"
}
習卷 6 則；考卷 2 則。


## 試訓並保存

以下以少量資料試行 10 步，需下載約數十 GB 的底座；若顯存不足或執行階段中斷便無法完成。完成後請下載壓縮檔；它只是 LoRA 附加權重，仍需相容的底座。

In [28]:
import subprocess
import shutil
from pathlib import Path

# 訓練腳本預設輸出路徑為 data/classical/model-adapter
output_dir = Path("data/classical/model-adapter")
if output_dir.exists():
    print(f"偵測到輸出目錄 {output_dir} 已存在，正在進行清理以利重新試訓...")
    shutil.rmtree(output_dir)

try:
    print("開始執行試訓...")
    result = subprocess.run(
        ["python", "-m", "training.train_classical", "--allow-small-dataset", "--max-steps", "10"],
        capture_output=True,
        text=True,
        check=True
    )
    print("--- 執行成功 ---")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("--- 執行失敗 ---")
    print("錯誤碼:", e.returncode)
    print("標準輸出 (stdout):\n", e.stdout)
    print("標準錯誤 (stderr):\n", e.stderr)

偵測到輸出目錄 data/classical/model-adapter 已存在，正在進行清理以利重新試訓...
開始執行試訓...
--- 執行成功 ---
習卷 6 則；考卷 2 則。
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.10: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.
Unsloth: Offloading embeddings to RAM to save 1.08 GB.
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Detected MoE model with per-expe

In [29]:
from pathlib import Path
import shutil
from google.colab import files
adapter = Path("data/classical/model-adapter")
assert (adapter / "adapter_config.json").exists(), "訓練尚未產生附加權重"
archive = shutil.make_archive("/content/classical-adapter", "zip", adapter)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 多輪模型評估

以下以同一組題目重複兩輪，記錄歷史、年代限制、未知問題、程式及文風的原始回答、耗時與是否達到輸出長度上限。採用確定性生成，重複輪次主要檢查穩定性，並不代表獨立樣本。結果需要人工對照來源、逐題審核；程式須另行執行。小樣本不能產生可靠的整體精度，也不能證明模型沒有 1899 年後的知識。此步驟會重新載入模型，可能需要較多 GPU 記憶體與時間。

In [48]:
import subprocess

try:
    print("使用正確的 evals 進入點開始執行模型多輪評估...")
    # 執行 gpt_oss.evals 模組，傳遞剛訓練好的 model-adapter
    result = subprocess.run(
        ["python", "-m", "gpt_oss.evals", "--adapter", "data/classical/model-adapter", "--task", "classical", "--output", "data/classical/evaluation.json"],
        capture_output=True,
        text=True,
        check=True
    )
    print("--- 執行成功 ---")
    print("標準輸出 (stdout):\n", result.stdout)
except subprocess.CalledProcessError as e:
    print("--- 執行失敗 ---")
    print("錯誤碼:", e.returncode)
    print("標準輸出 (stdout):\n", e.stdout)
    print("標準錯誤 (stderr):\n", e.stderr)

使用正確的 evals 進入點開始執行模型多輪評估...
--- 執行失敗 ---
錯誤碼: 2
標準輸出 (stdout):
 
標準錯誤 (stderr):
 usage: __main__.py [-h] [--model MODEL] [--reasoning-effort REASONING_EFFORT]
                   [--sampler {responses,chat_completions}]
                   [--base-url BASE_URL] [--eval EVAL]
                   [--temperature TEMPERATURE] [--n-threads N_THREADS]
                   [--debug] [--examples EXAMPLES]
__main__.py: error: unrecognized arguments: --adapter data/classical/model-adapter --task classical --output data/classical/evaluation.json



In [51]:
from pathlib import Path

classical_eval_script = Path("gpt_oss/evals/classical_game_eval.py")
if classical_eval_script.exists():
    print(f"--- 讀取古典評估腳本: {classical_eval_script} ---")
    content = classical_eval_script.read_text(encoding="utf-8", errors="ignore")
    print(content[:4000])
else:
    print(f"找不到古典評估腳本：{classical_eval_script}")

--- 讀取古典評估腳本: gpt_oss/evals/classical_game_eval.py ---
"""命學者製戲，存其原言，列疑以待覆核。"""

import argparse
import json
import os
import re
import urllib.error
import urllib.request
from pathlib import Path


GAME_PROMPT = """請製作一局可於瀏覽器遊玩的「九宮移數」之戲。
以一至八之籌置於九宮，留一空位。鄰空位之籌可移入，復歸順序則勝。
請依次交付：一、以繁體文言述規則與構思；二、以一個完整 html 程式碼區塊交付
可獨立開啟的成品，內含所需之 HTML、CSS、JavaScript，不借外部函式庫或網路資源；
三、以繁體文言列出驗法，明言哪些已驗、哪些未驗。
須有開局、重局、步數、勝負告示及觸控操作；開局必可解，且不得已是勝局。
說明、註釋與畫面文字皆用清楚之文言；自訂名稱取義於九宮與算籌。
現代保留字與介面依其原式。勿捏造此戲之古代出處。
請交付完整成品，不以省略號、待補段落或只有設計說明代替。
"""


def inspect_scroll(answer: str, response_status: str) -> tuple[list[str], str | None]:
    """只列可見之疑，不以字詞檢索斷定文章優劣。"""
    flags = []
    if response_status != "completed":
        flags.append(f"回覆未正常完成：{response_status}")
    blocks = re.findall(r"```html\s*\n([\s\S]*?)```", answer, re.I)
    game = blocks[0].strip() if len(blocks) == 1 else None
    if game is None:
        flags.append("未得唯一完整 html 區塊。")
    elif not re.search(r"</html\s*>", game, re.I):
        flags.append("HT

下載 JSON，逐題填寫 human_review 的欄位；null 表示尚未驗證。若輸出截斷，增加 max-new-tokens 並重試，注意模型上下文長度與 GPU 記憶體。保留 raw_output 以便檢查格式，請勿把其中的內部標記直接顯示給聊天使用者。

In [40]:
from google.colab import files
from pathlib import Path

target_file = Path("data/classical/evaluation.json")
if target_file.exists():
    files.download(str(target_file))
else:
    print(f"錯誤：找不到檔案 {target_file}。請確認前一個評估步驟 (ullFgCtKepsh) 是否已成功執行完畢並產生此檔案。")

錯誤：找不到檔案 data/classical/evaluation.json。請確認前一個評估步驟 (ullFgCtKepsh) 是否已成功執行完畢並產生此檔案。
